# neuron_methods

**15 runs · unmeasured wall clock on 2×T4 · nothing to edit.**

The professor-requested next step: compare ReDo with two published techniques that act directly on inactive neurons. SNR uses each unit's inter-firing-time history; ReGraMa uses normalized gradient magnitude. Five paired seeds per arm, on the established Permuted-MNIST setting. Runtime is left unestimated until the smoke test because SNR's reset count is data-adaptive.

## Before you run
1. Attach the code Dataset (the one built by `scripts/package_for_kaggle.py`).
   It bundles `mnist.npz`, so it is the only Dataset you need.
2. Accelerator → **GPU T4 × 2**.
3. Run All.

The config-count assertion in the "Pick the experiment" cell fails fast if the
Dataset is a stale version, rather than silently running the wrong sweep.

## When it finishes
- **`extract.zip`** — download this. A few MB; everything the analysis needs.
- **`runs.zip`** — push to a versioned Dataset, do not download. It holds the
  per-neuron log, which cannot be rebuilt after the fact (CLAUDE.md §5.4).

This notebook contains no logic: it imports from `src/` and calls one function.

In [ ]:
import os, sys, glob, shutil, subprocess
from pathlib import Path

# Locate the attached datasets by CONTENT, not by name. Kaggle slugifies dataset
# titles and may nest an upload a level or two deeper than you expect; two
# sessions were lost to exactly that.
_repo_hits = sorted(glob.glob("/kaggle/input/**/src/train.py", recursive=True))
assert _repo_hits, "code dataset not attached (no src/train.py under /kaggle/input)"
REPO = str(Path(_repo_hits[0]).parents[1])

# MNIST ships inside the code dataset; CIFAR-10 is attached separately and comes
# in whichever layout the public dataset happens to use -- src/data.py reads all
# three (npz, extracted pickle batches, or the official tarball).
_mnist = sorted(glob.glob("/kaggle/input/**/mnist.npz", recursive=True))
_cifar = sorted(
    glob.glob("/kaggle/input/**/cifar10.npz", recursive=True)
    + glob.glob("/kaggle/input/**/cifar-10-batches-py", recursive=True)
    + glob.glob("/kaggle/input/**/cifar-10-python.tar.gz", recursive=True)
)
DATA_MNIST = str(Path(_mnist[0]).parent) if _mnist else None
DATA_CIFAR = str(Path(_cifar[0]).parent) if _cifar else None
DATA = DATA_MNIST  # the next cell overrides this for CIFAR experiments
RUNS = "/kaggle/working/runs"

# Must be set before any CUDA context exists (see src/config.set_determinism).
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
sys.path.insert(0, REPO)

import torch
print("REPO       ", REPO)
print("DATA_MNIST ", DATA_MNIST)
print("DATA_CIFAR ", DATA_CIFAR)
print(torch.__version__, torch.cuda.device_count(), "GPU(s)")

## Always run the tests first

Twenty seconds of testing beats losing a 13-hour sweep (CLAUDE.md §8). This runs
on synthetic data and needs no GPU.

In [ ]:
# -p no:cacheprovider: the code dataset is mounted read-only, so pytest cannot
# write .pytest_cache and would fail on that alone.
subprocess.run([sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "tests/"],
               cwd=REPO, check=True)

## Pick the experiment, then smoke-test one config

`EXPERIMENT` selects a directory under `configs/`. The protocol's gate-failure
responses must be tried **in order** (§A.4): `gate_hi` (raise the learning rate),
then `gate_long` (400 tasks), then `gate_narrow` (width 200).

Expect ~14 min per 200-task run on a T4.

In [ ]:
import time

# --- pre-filled for this job; nothing to edit ------------------------------
EXPERIMENT = "neuron_methods"
CONFIG_GLOBS = ["*.json"]
RUN_PATTERN = "methods_*"
IS_GATE = False
EXPECTED_RUNS = 15
DATA = DATA_MNIST
assert DATA, "MNIST not found in the code dataset"
# ---------------------------------------------------------------------------

configs = sorted(
    {p for g in CONFIG_GLOBS for p in glob.glob(f"{REPO}/configs/{EXPERIMENT}/{g}")}
)
assert configs, f"no configs matching {CONFIG_GLOBS} in {REPO}/configs/{EXPERIMENT}"
assert len(configs) == EXPECTED_RUNS, (
    f"expected {EXPECTED_RUNS} configs, found {len(configs)}. The code Dataset "
    "is probably an older version -- re-upload before running the sweep."
)
print(f"{EXPERIMENT}: {len(configs)} configs -- as expected")
_smoke_matches = glob.glob(f"{REPO}/configs/{EXPERIMENT}/methods_snr_eta0p08_lr0p1_s0.json")
assert len(_smoke_matches) == 1, "expected exactly one configured smoke-test run"
smoke_config = _smoke_matches[0]
print(f"smoke-test config: {Path(smoke_config).name}")

# Smoke-test as a SUBPROCESS, never in this kernel. A Trainer run here keeps its
# CUDA allocations for the life of the session, and the next cell then launches
# children onto the same GPUs which OOM. That killed a Setting 2 session: a CNN
# probing 2048 images holds far more memory than the MLPs ever did.
_t0 = time.perf_counter()
subprocess.run(
    [sys.executable, "-m", "src.train", "--config", smoke_config,
     "--runs-root", RUNS, "--data-root", DATA, "--device", "cuda"],
    cwd=REPO, check=True,
)
_per_run = time.perf_counter() - _t0
print(f"\none run: {_per_run:.0f}s")
print(f"{len(configs)} runs on 2 GPUs = {_per_run * len(configs) / 2 / 3600:.2f} h")

## The full gate: 15 runs, two per GPU pass

`launch_pair.py` runs two configs at a time via `CUDA_VISIBLE_DEVICES` and stops
launching new work at `--budget-hours`, so the 12-hour kill never lands mid-run.
Anything interrupted resumes from its own checkpoint by `run_id`.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/launch_pair.py", *configs,
     "--runs-root", RUNS, "--data-root", DATA, "--budget-hours", "10.5"],
    cwd=REPO, check=True,
)

## Evaluate the gate

Applies the frozen criterion in `configs/analysis_plan.json`. If the accuracy
gate passes but dead units stay flat, **stop and report it** — that dissociation
is itself a result and changes the paper (protocol §A.4).

In [ ]:
if IS_GATE:
    subprocess.run(
        [sys.executable, "-m", "src.analysis.gate", "--runs-root", RUNS,
         "--pattern", RUN_PATTERN],
        cwd=REPO, check=False,
    )
else:
    print(f"{EXPERIMENT} is not a gate experiment; the frozen gate criterion does "
          "not apply to it. Analysis happens off-Kaggle from extract.zip.")

## Persist results

Three artifacts, increasing in size. All stay attached to this notebook version
on Kaggle, so nothing is lost by downloading them later.

| file | size | what it is | download? |
|---|---|---|---|
| **`extract.zip`** | a few MB | per-task + per-layer metrics, and the recycled-set composition table | **always** — this is C1, C2, C3 and the gate |
| **`c4.zip`** | ~5 MB/run | slim per-neuron table: death flag, magnitude, recycling flag, weight and gradient norms | when doing the C4 survival analysis |
| **`runs.zip`** | ~27 MB/run | everything, incl. the full 23-column per-neuron log and checkpoints | archival only — push to a versioned Dataset |

`c4.zip` omits `sokar_score`: it was 30% of the file and is exactly recomputable
from `mean_abs_act` via `src.analysis.load.add_sokar_score`.

In [ ]:
subprocess.run([sys.executable, "scripts/update_ledger.py", "--runs-root", RUNS,
                "--out", "/kaggle/working/LEDGER.md"], cwd=REPO, check=True)

# 1. Small -- always download this one.
subprocess.run([sys.executable, "scripts/make_analysis_extract.py",
                "--runs-root", RUNS, "--out", "/kaggle/working/extract", "--zip"],
               cwd=REPO, check=True)

# 2. Medium -- the C4 survival dataset, as its OWN zip so it never bloats (1).
subprocess.run([sys.executable, "scripts/make_analysis_extract.py",
                "--runs-root", RUNS, "--out", "/kaggle/working/c4",
                "--with-c4", "--zip"],
               cwd=REPO, check=True)

# 3. Large -- archival. Push to a versioned Dataset; do not download.
print(shutil.make_archive("/kaggle/working/runs", "zip", RUNS))